In [1]:
import os
import json
import pickle
import random
from collections import Counter, defaultdict

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from scipy.sparse import csr_matrix

In [2]:
# CONFIG
DATA_PATH = "/home/brocchio/teams/z2/data/steam/cleaned/steam_reviews_full_with_genres.parquet"

MIN_REVIEWS = 5
TARGET_USERS = 20000
SEEDS = [232, 777, 2026]

TRAIN_END_YEAR = 2022  # train: year < cutoff, test: year >= cutoff

MIN_TRAIN_POS = 5
K = 10

MAX_TRAIN_PER_USER = 200
MAX_TEST_PER_USER = 200

# ALS defaults
ALS_DEFAULT = dict(factors=32, regularization=0.05, iterations=20, alpha=20)

# BM25 defaults
BM25_DEFAULT = dict(K1=1.2, B=0.75)


## 1) Eligibility filtering (users with ≥ MIN_REVIEWS reviews)

We first scan the full dataset to count reviews per user.
Then we keep only users with at least `MIN_REVIEWS` interactions.

- **Pass 1**: count user reviews
- **Pass 2**: sanity check remaining rows and remaining unique users


In [3]:
pf = pq.ParquetFile(DATA_PATH)
user_counts = Counter()

for rg in range(pf.num_row_groups):
    tbl = pf.read_row_group(rg, columns=["author_steamid"])
    user_counts.update(tbl["author_steamid"].to_pylist())
    if rg % 50 == 0:
        print(f"pass1 counted rg {rg}/{pf.num_row_groups}")

eligible_users = {u for u, c in user_counts.items() if c >= MIN_REVIEWS}
print("Total unique users (full):", len(user_counts))
print(f"Users with >= {MIN_REVIEWS} reviews:", len(eligible_users))


pass1 counted rg 0/165
pass1 counted rg 50/165
pass1 counted rg 100/165
pass1 counted rg 150/165
Total unique users (full): 37918537
Users with >= 5 reviews: 5394398


In [4]:
remaining_reviews = 0
remaining_users_seen = set()

for rg in range(pf.num_row_groups):
    tbl = pf.read_row_group(rg, columns=["author_steamid"])
    users = tbl["author_steamid"].to_pylist()
    keep_mask = [u in eligible_users for u in users]
    n_keep = sum(keep_mask)

    remaining_reviews += n_keep
    for u, keep in zip(users, keep_mask):
        if keep:
            remaining_users_seen.add(u)

    if rg % 50 == 0:
        print(f"pass2 scanned rg {rg}/{pf.num_row_groups}")

print("Remaining reviews (rows):", remaining_reviews)
print("Remaining unique users:", len(remaining_users_seen))
print("Avg reviews per remaining user:", remaining_reviews / max(1, len(remaining_users_seen)))


pass2 scanned rg 0/165
pass2 scanned rg 50/165
pass2 scanned rg 100/165
pass2 scanned rg 150/165
Remaining reviews (rows): 62713168
Remaining unique users: 5394398
Avg reviews per remaining user: 11.625610123687574


## 2) Sample users

We sample `TARGET_USERS` users from the eligible pool using a fixed seed.
This keeps experiments reproducible.


In [5]:
def sample_users(eligible_users: set, target_users: int, seed: int) -> set:
    random.seed(seed)
    eligible_list = list(eligible_users)
    n = min(target_users, len(eligible_list))
    return set(random.sample(eligible_list, n))

seed = SEEDS[0]
sampled_users = sample_users(eligible_users, TARGET_USERS, seed)
print("Sampled users:", len(sampled_users), "| seed:", seed)


Sampled users: 20000 | seed: 232


## 3) Train/test split (time-based)

- **Train**: interactions where `year(timestamp_created) < TRAIN_END_YEAR`
- **Test**: interactions where `year(timestamp_created) >= TRAIN_END_YEAR` AND `voted_up=True`

We keep test positives only (ranking evaluation).

Evaluate whether the model can
recommend *future positive* games for each user.


In [6]:
def build_train_test_pos_only(
    data_path: str,
    sampled_users: set,
    train_end_year: int,
    max_train_per_user: int,
    max_test_per_user: int,
):
    pf = pq.ParquetFile(data_path)
    train_items_by_user = defaultdict(list)   # positives in train
    test_items_by_user = defaultdict(set)     # positives in test

    for rg in range(pf.num_row_groups):
        tbl = pf.read_row_group(
            rg,
            columns=["author_steamid", "appid", "voted_up", "timestamp_created"]
        )

        users = tbl["author_steamid"].to_pylist()
        keep_mask = pa.array([u in sampled_users for u in users])
        tbl = tbl.filter(keep_mask)
        if tbl.num_rows == 0:
            continue

        years = pc.year(tbl["timestamp_created"])
        is_pos = pc.equal(tbl["voted_up"], True)

        train_mask = pc.and_(pc.less(years, train_end_year), is_pos)
        test_mask  = pc.and_(pc.greater_equal(years, train_end_year), is_pos)

        train_tbl = tbl.filter(train_mask)
        test_tbl  = tbl.filter(test_mask)

        for u, a in zip(train_tbl["author_steamid"].to_pylist(), train_tbl["appid"].to_pylist()):
            if len(train_items_by_user[u]) < max_train_per_user:
                train_items_by_user[u].append(a)

        for u, a in zip(test_tbl["author_steamid"].to_pylist(), test_tbl["appid"].to_pylist()):
            if len(test_items_by_user[u]) < max_test_per_user:
                test_items_by_user[u].add(a)

        if rg % 25 == 0:
            print(f"[split] scanned row group {rg}/{pf.num_row_groups}")

    users_final = [u for u, items in train_items_by_user.items() if len(items) >= MIN_TRAIN_POS]
    n_eval = sum(1 for u in users_final if len(test_items_by_user.get(u, set())) > 0)

    print("users_final (>= MIN_TRAIN_POS train positives):", len(users_final))
    print("users_final with any test positives:", n_eval)
    return train_items_by_user, test_items_by_user, users_final

train_items_by_user, test_items_by_user, users_final = build_train_test_pos_only(
    DATA_PATH, sampled_users, TRAIN_END_YEAR, MAX_TRAIN_PER_USER, MAX_TEST_PER_USER
)


[split] scanned row group 0/165
[split] scanned row group 25/165
[split] scanned row group 50/165
[split] scanned row group 75/165
[split] scanned row group 100/165
[split] scanned row group 125/165
[split] scanned row group 150/165
users_final (>= MIN_TRAIN_POS train positives): 11004
users_final with any test positives: 6281


## 4) Build sparse user–item matrix (CSR)

We build `X_user_item` where:
- rows = users
- cols = games (appid)
- entries = 1 (implicit positive interaction)

This is the matrix used by implicit ALS.


In [7]:
def build_csr_binary(train_items_by_user: dict, users_final: list):
    user2idx = {u: i for i, u in enumerate(users_final)}

    item_set = set()
    for u in users_final:
        item_set.update(train_items_by_user[u])
    items = list(item_set)

    item2idx = {a: i for i, a in enumerate(items)}
    idx2item = {i: a for a, i in item2idx.items()}

    rows, cols, vals = [], [], []
    for u in users_final:
        ui = user2idx[u]
        for a in train_items_by_user[u]:
            ii = item2idx.get(a)
            if ii is not None:
                rows.append(ui); cols.append(ii); vals.append(1.0)

    X = csr_matrix((vals, (rows, cols)), shape=(len(users_final), len(items))).tocsr()
    print("X_user_item shape:", X.shape, "| nnz:", X.nnz)
    return X, user2idx, item2idx, idx2item

X_user_item, user2idx, item2idx, idx2item = build_csr_binary(train_items_by_user, users_final)


X_user_item shape: (11004, 11281) | nnz: 114120


## 5) Evaluation metrics

For each user:
- generate top-K recommendations
- compare against that user’s **test positives**

Metrics:
- **Hit Rate @K (HR@K):** fraction of users with ≥1 hit in top K
- **Recall @K:** total hits / total true test items
- **Precision @K:** total hits / (users_evaluated * K)


In [8]:
def eval_topk_implicit(model, X, users_final, user2idx, idx2item, test_items_by_user, K=10):
    hits_users = 0
    hits_total = 0
    true_total = 0
    users_evaluated = 0

    for u in users_final:
        true = test_items_by_user.get(u, set())
        if not true:
            continue

        users_evaluated += 1
        ui = int(user2idx[u])

        ids, scores = model.recommend(
            np.array([ui]),
            X[[ui]],
            N=K,
            filter_already_liked_items=True,
        )

        rec_ids = ids[0]
        rec_items = {idx2item[int(i)] for i in rec_ids}

        inter = true.intersection(rec_items)
        hits_total += len(inter)
        true_total += len(true)
        if inter:
            hits_users += 1

    hr = hits_users / users_evaluated if users_evaluated else 0.0
    recall = hits_total / true_total if true_total else 0.0
    precision = hits_total / (users_evaluated * K) if users_evaluated else 0.0
    return hr, recall, precision, users_evaluated


## 6) Popularity baseline

Recommend the globally most-popular games from training data (excluding games the user already has in train).
This is a strong baseline for implicit feedback.


In [9]:
def eval_popularity(users_final, train_items_by_user, test_items_by_user, top_appids, K=10):
    hits_users = 0
    hits_total = 0
    true_total = 0
    users_evaluated = 0

    for u in users_final:
        true = test_items_by_user.get(u, set())
        if not true:
            continue
        users_evaluated += 1

        seen = set(train_items_by_user.get(u, []))
        recs = [a for a in top_appids if a not in seen][:K]

        inter = [a for a in recs if a in true]
        hits_total += len(inter)
        true_total += len(true)
        if inter:
            hits_users += 1

    hr = hits_users / users_evaluated if users_evaluated else 0.0
    recall = hits_total / true_total if true_total else 0.0
    precision = hits_total / (users_evaluated * K) if users_evaluated else 0.0
    return hr, recall, precision, users_evaluated

pop_counter = Counter()
for u in users_final:
    pop_counter.update(train_items_by_user[u])

top_appids = [a for a, _ in pop_counter.most_common(500)]
hrp, recp, precp, n_pop = eval_popularity(users_final, train_items_by_user, test_items_by_user, top_appids, K=K)
print(f"Popularity | HR@{K}={hrp:.6f} Recall@{K}={recp:.6f} Prec@{K}={precp:.6f} | Users={n_pop}")


Popularity | HR@10=0.118134 Recall@10=0.033682 Prec@10=0.014616 | Users=6281


## 7) ALS baseline (implicit feedback)

We train **Implicit ALS** on the binary interaction matrix.

Key hyperparameters:
- factors (embedding dimension)
- regularization
- alpha (confidence scaling)
- iterations


In [10]:
from implicit.als import AlternatingLeastSquares

cfg = ALS_DEFAULT.copy()
m_als = AlternatingLeastSquares(
    factors=cfg["factors"],
    regularization=cfg["regularization"],
    iterations=cfg["iterations"],
)
m_als.fit(X_user_item * cfg["alpha"])

hr, rec, prec, n_eval = eval_topk_implicit(m_als, X_user_item, users_final, user2idx, idx2item, test_items_by_user, K=K)
print(f"ALS | HR@{K}={hr:.6f} Recall@{K}={rec:.6f} Prec@{K}={prec:.6f} | Users={n_eval}")


/home/brocchio/.local/lib/python3.11/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 128 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
or set the environment variable OPENBLAS_NUM_THREADS to 128 or lower


OpenBLAS : Program is Terminated. Because you tried to allocate too many memory regions.
This library was built to support a maximum of 256 threads - either rebuild OpenBLAS
with a larger NUM_THREADS value or set the environment variable OPENBLAS_NUM_THREADS to
a sufficiently small number. This error typically occurs when the software that relies on
OpenBLAS calls BLAS functions from many threads in parallel, or when your computer has more
cpu cores than what OpenBLAS was configured to handle.
OpenBLAS : Program is Terminated. Because you tried to allocate too many memory regions.
OpenBLAS : Program is Terminated. Because you tried to allocate too many memory regions.
This library was built to support a maximum of 256 threads - either rebuild OpenBLAS
with a larger NUM_THREADS value or set the environment variable OPENBLAS_NUM_THREADS to
a sufficiently small number. This error typically occurs when the software that relies on
OpenBLAS calls BLAS functions from many threads in parallel,

## 7.5 Hyperparameter sweep (Raw ALS)

We tune the main ALS hyperparameters on the same sampled users + time-based split:

- `factors` (embedding dimension)
- `regularization`
- `alpha` (confidence scaling for implicit feedback)

We select the best configuration by **HR@K**, and report Recall@K and Precision@K as secondary metrics.

> Note: this is not an exhaustive search; it's a compact sweep to justify our final hyperparameters.


In [25]:
import itertools
from implicit.als import AlternatingLeastSquares

def train_eval_raw_als(X, users_final, user2idx, idx2item, test_items_by_user, cfg, K=10):
    m = AlternatingLeastSquares(
        factors=cfg["factors"],
        regularization=cfg["regularization"],
        iterations=cfg["iterations"],
    )
    m.fit(X * cfg["alpha"])
    hr, rec, prec, n_eval = eval_topk_implicit(
        m, X, users_final, user2idx, idx2item, test_items_by_user, K=K
    )
    return hr, rec, prec, n_eval, m

# Compact grid (adjust if we have time)
grid = {
    "factors": [16, 32, 64],
    "regularization": [0.01, 0.05, 0.1],
    "alpha": [10, 20, 40],
    "iterations": [ALS_DEFAULT["iterations"]],
}

raw_sweep_rows = []
best = None  # (hr, cfg, metrics, model)

for factors, reg, alpha, iters in itertools.product(
    grid["factors"], grid["regularization"], grid["alpha"], grid["iterations"]
):
    cfg = dict(factors=factors, regularization=reg, alpha=alpha, iterations=iters)
    hr, rec, prec, n_eval, m = train_eval_raw_als(
        X_user_item, users_final, user2idx, idx2item, test_items_by_user, cfg, K=K
    )
    raw_sweep_rows.append({**cfg, "hr": hr, "recall": rec, "precision": prec, "users": n_eval})

    if best is None or hr > best[0]:
        best = (hr, cfg, (rec, prec, n_eval), m)

best_hr, best_cfg, (best_rec, best_prec, best_users), best_model_raw = best
print("Best RAW ALS by HR@{}:".format(K))
print(best_cfg)
print(f"HR@{K}={best_hr:.6f} Recall@{K}={best_rec:.6f} Prec@{K}={best_prec:.6f} | Users={best_users}")


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Best RAW ALS by HR@10:
{'factors': 16, 'regularization': 0.01, 'alpha': 20, 'iterations': 20}
HR@10=0.175088 Recall@10=0.052157 Prec@10=0.022210 | Users=6254


## 8) BM25-weighted ALS (final candidate)

We apply BM25 weighting to the user–item matrix before ALS.

**Intuition:**
- downweights extremely popular games (less informative)
- normalizes heavy users
- emphasizes more discriminative interactions

Then ALS learns embeddings from the reweighted matrix.


In [27]:
from implicit.nearest_neighbours import bm25_weight

bm = BM25_DEFAULT.copy()
X_bm25 = bm25_weight(X_user_item, K1=bm["K1"], B=bm["B"]).tocsr()

cfg = ALS_DEFAULT.copy()
m_bm25 = AlternatingLeastSquares(
    factors=cfg["factors"],
    regularization=cfg["regularization"],
    iterations=cfg["iterations"],
)
m_bm25.fit(X_bm25 * cfg["alpha"])

hr, rec, prec, n_eval = eval_topk_implicit(m_bm25, X_bm25, users_final, user2idx, idx2item, test_items_by_user, K=K)
print(f"BM25+ALS | HR@{K}={hr:.6f} Recall@{K}={rec:.6f} Prec@{K}={prec:.6f} | Users={n_eval}")


  0%|          | 0/20 [00:00<?, ?it/s]

BM25+ALS | HR@10=0.166933 Recall@10=0.048515 Prec@10=0.020659 | Users=6254


## 8.5) Hyperparameter sweep (BM25 + ALS)

We tune BM25 reweighting parameters:

- `K1`: term-frequency saturation
- `B`: length normalization

and (optionally) ALS hyperparameters (`factors`, `regularization`, `alpha`) on the same
train/test split.

We select the best configuration by HR@K and report Recall@K and Precision@K.


In [28]:
from implicit.nearest_neighbours import bm25_weight
from implicit.als import AlternatingLeastSquares
import itertools
import pandas as pd
import numpy as np

def train_eval_bm25_als(X_base, users_final, user2idx, idx2item, test_items_by_user, cfg, bm25_cfg, K=10):
    # Reweight interactions with BM25
    Xw = bm25_weight(X_base, K1=bm25_cfg["K1"], B=bm25_cfg["B"]).tocsr()

    # Train ALS
    m = AlternatingLeastSquares(
        factors=cfg["factors"],
        regularization=cfg["regularization"],
        iterations=cfg["iterations"],
    )
    m.fit(Xw * cfg["alpha"])

    # Evaluate
    hr, rec, prec, n_eval = eval_topk_implicit(
        m, Xw, users_final, user2idx, idx2item, test_items_by_user, K=K
    )
    return hr, rec, prec, n_eval, m


In [29]:
# Keep ALS fixed
ALS_FIXED = ALS_DEFAULT.copy()

bm25_grid = {
    "K1": [0.5, 1.2, 2.0, 3.0],
    "B":  [0.2, 0.5, 0.75, 0.9],
}

bm25_rows = []
best = None  # (hr, bm25_cfg, metrics, model)

for K1, B in itertools.product(bm25_grid["K1"], bm25_grid["B"]):
    bm_cfg = {"K1": float(K1), "B": float(B)}
    hr, rec, prec, n_eval, m = train_eval_bm25_als(
        X_user_item, users_final, user2idx, idx2item, test_items_by_user,
        cfg=ALS_FIXED, bm25_cfg=bm_cfg, K=K
    )
    bm25_rows.append({"K1": K1, "B": B, "hr": hr, "recall": rec, "precision": prec, "users": n_eval})

    if best is None or hr > best[0]:
        best = (hr, bm_cfg, (rec, prec, n_eval), m)

best_hr, best_bm25_cfg, (best_rec, best_prec, best_users), best_bm25_model = best
print("Best BM25 params (ALS fixed) by HR@{}:".format(K))
print(best_bm25_cfg)
print(f"HR@{K}={best_hr:.6f} Recall@{K}={best_rec:.6f} Prec@{K}={best_prec:.6f} | Users={best_users}")

df_bm25 = pd.DataFrame(bm25_rows).sort_values(["hr", "recall", "precision"], ascending=False)
df_bm25.head(12)


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Best BM25 params (ALS fixed) by HR@10:
{'K1': 2.0, 'B': 0.9}
HR@10=0.162936 Recall@10=0.047614 Prec@10=0.020275 | Users=6254


,K1,B,hr,recall,precision,users
11,2.0,0.90,0.162936,0.047614,0.020275,6254
13,3.0,0.50,0.162296,0.047163,0.020083,6254
2,0.5,0.75,0.160377,0.046788,0.019923,6254
3,0.5,0.90,0.160217,0.046900,0.019971,6254
10,2.0,0.75,0.160058,0.046562,0.019827,6254
15,3.0,0.90,0.159418,0.046337,0.019731,6254
5,1.2,0.50,0.159098,0.047163,0.020083,6254
6,1.2,0.75,0.159098,0.045924,0.019555,6254
14,3.0,0.75,0.157979,0.046600,0.019843,6254
8,2.0,0.20,0.157979,0.046525,0.019811,6254


In [30]:
als_grid = {
    "factors": [16, 32, 64],
    "regularization": [0.01, 0.05],
    "alpha": [10, 20, 40],
    "iterations": [ALS_DEFAULT["iterations"]],
}

bm25_grid_small = {
    "K1": [0.5, 1.2, 2.0],
    "B":  [0.5, 0.75, 0.9],
}

rows = []
best = None  # (hr, cfg, bm_cfg, metrics, model)

for factors, reg, alpha, iters in itertools.product(
    als_grid["factors"], als_grid["regularization"], als_grid["alpha"], als_grid["iterations"]
):
    cfg = {"factors": factors, "regularization": reg, "alpha": alpha, "iterations": iters}

    for K1, B in itertools.product(bm25_grid_small["K1"], bm25_grid_small["B"]):
        bm_cfg = {"K1": float(K1), "B": float(B)}

        hr, rec, prec, n_eval, m = train_eval_bm25_als(
            X_user_item, users_final, user2idx, idx2item, test_items_by_user,
            cfg=cfg, bm25_cfg=bm_cfg, K=K
        )

        rows.append({**cfg, **bm_cfg, "hr": hr, "recall": rec, "precision": prec, "users": n_eval})

        if best is None or hr > best[0]:
            best = (hr, cfg, bm_cfg, (rec, prec, n_eval), m)

best_hr, best_cfg, best_bm_cfg, (best_rec, best_prec, best_users), best_model = best
print("Best JOINT BM25+ALS by HR@{}:".format(K))
print("ALS cfg:", best_cfg)
print("BM25 cfg:", best_bm_cfg)
print(f"HR@{K}={best_hr:.6f} Recall@{K}={best_rec:.6f} Prec@{K}={best_prec:.6f} | Users={best_users}")

df_joint = pd.DataFrame(rows).sort_values(["hr", "recall", "precision"], ascending=False)
df_joint.head(15)


  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [55]:
from implicit.nearest_neighbours import bm25_weight
from implicit.als import AlternatingLeastSquares

# Best params from grid
BEST_ALS = {"factors": 16, "regularization": 0.01, "alpha": 10, "iterations": 20}
BEST_BM25 = {"K1": 0.5, "B": 0.75}

# Build BM25-weighted matrix
X_bm25_best = bm25_weight(X_user_item, K1=BEST_BM25["K1"], B=BEST_BM25["B"]).tocsr()

# Train ALS on BM25-weighted matrix
m_bm25_best = AlternatingLeastSquares(
    factors=BEST_ALS["factors"],
    regularization=BEST_ALS["regularization"],
    iterations=BEST_ALS["iterations"],
)
m_bm25_best.fit(X_bm25_best * BEST_ALS["alpha"])


  0%|          | 0/20 [00:00<?, ?it/s]

In [58]:
hr_bm25_best, rec_bm25_best, prec_bm25_best, n_bm25_best = eval_topk_implicit(
    m_bm25_best, X_bm25_best, users_final, user2idx, idx2item, test_items_by_user, K=K
)

print(f"BM25+ALS BEST | HR@{K}={hr_bm25_best:.6f} Recall@{K}={rec_bm25_best:.6f} Prec@{K}={prec_bm25_best:.6f} | Users={n_bm25_best}")


BM25+ALS BEST | HR@10=0.167413 Recall@10=0.049942 Prec@10=0.021266 | Users=6254


In [11]:
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight
import pickle
from datetime import datetime

# Raw ALS default
ALS_DEFAULT_CFG = {
    "factors": ALS_DEFAULT["factors"],
    "regularization": ALS_DEFAULT["regularization"],
    "alpha": ALS_DEFAULT["alpha"],
    "iterations": ALS_DEFAULT["iterations"],
}

# Raw ALS tuned
if "BEST_ALS" not in globals():
    BEST_ALS = {"factors": 16, "regularization": 0.01, "alpha": 20, "iterations": ALS_DEFAULT["iterations"]}

BM25_DEFAULT_CFG = {"K1": 1.2, "B": 0.75}


if "BEST_BM25" not in globals():
    BEST_BM25 = {"K1": 0.5, "B": 0.75}

In [12]:
def fit_raw_als(X, cfg):
    m = AlternatingLeastSquares(
        factors=cfg["factors"],
        regularization=cfg["regularization"],
        iterations=cfg["iterations"],
    )
    m.fit(X * cfg["alpha"])
    return m

def fit_bm25_als(X_raw, als_cfg, bm25_cfg):
    X_bm = bm25_weight(X_raw, K1=bm25_cfg["K1"], B=bm25_cfg["B"]).tocsr()
    m = AlternatingLeastSquares(
        factors=als_cfg["factors"],
        regularization=als_cfg["regularization"],
        iterations=als_cfg["iterations"],
    )
    m.fit(X_bm * als_cfg["alpha"])
    return m, X_bm

In [ ]:
# 1) Raw ALS (default)
m_als_default = fit_raw_als(X_user_item, ALS_DEFAULT_CFG)

# 2) Raw ALS (tuned)
m_als_tuned = fit_raw_als(X_user_item, BEST_ALS)

# 3) BM25+ALS (default BM25 + default ALS)
m_bm25_default, X_bm25_default = fit_bm25_als(X_user_item, ALS_DEFAULT_CFG, BM25_DEFAULT_CFG)

# 4) BM25+ALS (tuned BM25 + tuned ALS)
m_bm25_tuned, X_bm25_tuned = fit_bm25_als(X_user_item, BEST_ALS, BEST_BM25)

print("Trained: raw ALS default, raw ALS tuned, BM25+ALS default, BM25+ALS tuned")
print("X shapes:", X_user_item.shape, X_bm25_default.shape, X_bm25_tuned.shape)

In [14]:
def eval_model(name, model, X_eval):
    hr, rec, prec, n_eval = eval_topk_implicit(
        model, X_eval, users_final, user2idx, idx2item, test_items_by_user, K=K
    )
    return {"name": name, "hr": float(hr), "recall": float(rec), "precision": float(prec), "users": int(n_eval)}

metrics = {}
metrics["ALS_default"] = eval_model("ALS_default", m_als_default, X_user_item)
metrics["ALS_tuned"] = eval_model("ALS_tuned", m_als_tuned, X_user_item)
metrics["BM25_default"] = eval_model("BM25_default", m_bm25_default, X_bm25_default)
metrics["BM25_tuned"] = eval_model("BM25_tuned", m_bm25_tuned, X_bm25_tuned)

metrics

{'ALS_default': {'name': 'ALS_default',
  'hr': 0.1666932017194714,
  'recall': 0.04920198128783709,
  'precision': 0.021350103486705938,
  'users': 6281},
 'ALS_tuned': {'name': 'ALS_tuned',
  'hr': 0.1692405667887279,
  'recall': 0.04956888644285452,
  'precision': 0.02150931380353447,
  'users': 6281},
 'BM25_default': {'name': 'BM25_default',
  'hr': 0.15507084859098869,
  'recall': 0.045643001284168046,
  'precision': 0.019805763413469193,
  'users': 6281},
 'BM25_tuned': {'name': 'BM25_tuned',
  'hr': 0.16828530488775673,
  'recall': 0.04909190974133187,
  'precision': 0.02130234039165738,
  'users': 6281}}

In [ ]:
artifacts = {
    "created_at": datetime.now().isoformat(),
    "K": K,
    "mappings": {
        "user2idx": user2idx,
        "idx2item": idx2item,
    },
    "configs": {
        "ALS_DEFAULT_CFG": ALS_DEFAULT_CFG,
        "BEST_ALS": BEST_ALS,
        "BM25_DEFAULT_CFG": BM25_DEFAULT_CFG,
        "BEST_BM25": BEST_BM25,
    },
    "models": {
        "ALS_default": m_als_default,
        "ALS_tuned": m_als_tuned,
        "BM25_default": m_bm25_default,
        "BM25_tuned": m_bm25_tuned,
    },
    # NOTE: we do NOT pickle the CSR matrices to keep file size reasonable
    "metrics": metrics if "metrics" in globals() else None,
}

with open("Brandon_all_models.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Saved all_models.pkl")

## 9) Test: TF-IDF-weighted ALS

Similar idea to BM25 but uses TF-IDF weighting on the interaction matrix.
Often underperforms BM25 in recommender settings, but useful as an ablation.


In [32]:
from implicit.nearest_neighbours import tfidf_weight

X_tfidf = tfidf_weight(X_user_item).tocsr()

cfg = ALS_DEFAULT.copy()
m_tfidf = AlternatingLeastSquares(
    factors=cfg["factors"],
    regularization=cfg["regularization"],
    iterations=cfg["iterations"],
)
m_tfidf.fit(X_tfidf * cfg["alpha"])

hr, rec, prec, n_eval = eval_topk_implicit(m_tfidf, X_tfidf, users_final, user2idx, idx2item, test_items_by_user, K=K)
print(f"TFIDF+ALS | HR@{K}={hr:.6f} Recall@{K}={rec:.6f} Prec@{K}={prec:.6f} | Users={n_eval}")


  0%|          | 0/20 [00:00<?, ?it/s]

TFIDF+ALS | HR@10=0.153981 Recall@10=0.045736 Prec@10=0.019476 | Users=6254


## 10) Feature-weighted ALS: playtime-based confidence

Raw ALS uses a binary implicit signal (positive interaction = 1).

We test whether incorporating interaction *strength* improves recommendations by weighting each positive train interaction by:

- `w = log1p(playtime_at_review)`, capped to limit outliers

Evaluation remains the same: future positive (voted_up=True) test items.


In [37]:
PLAYTIME_COL = "author_playtime_at_review"  # adjust if parquet uses a different name
PLAYTIME_CAP = 8.0

def build_train_test_playtime_weights(
    data_path: str,
    sampled_users: set,
    train_end_year: int,
    max_train_per_user: int,
    max_test_per_user: int,
    playtime_col: str,
    cap: float = 8.0,
):
    pf = pq.ParquetFile(data_path)
    train_w_by_user = defaultdict(dict)
    test_items_by_user = defaultdict(set)
    for rg in range(pf.num_row_groups):
        tbl = pf.read_row_group(
            rg,
            columns=["author_steamid", "appid", "voted_up", "timestamp_created", playtime_col],
        )
        users = tbl["author_steamid"].to_pylist()
        keep_mask = pa.array([u in sampled_users for u in users])
        tbl = tbl.filter(keep_mask)
        if tbl.num_rows == 0:
            continue

        years = pc.year(tbl["timestamp_created"])
        is_pos = pc.equal(tbl["voted_up"], True)

        train_tbl = tbl.filter(pc.and_(pc.less(years, train_end_year), is_pos))
        test_tbl  = tbl.filter(pc.and_(pc.greater_equal(years, train_end_year), is_pos))

        # Train weights
        u_list = train_tbl["author_steamid"].to_pylist()
        a_list = train_tbl["appid"].to_pylist()
        p_list = train_tbl[playtime_col].to_pylist()

        for u, a, p in zip(u_list, a_list, p_list):
            # cap unique items per user
            if (a not in train_w_by_user[u]) and (len(train_w_by_user[u]) >= max_train_per_user):
                continue

            p = 0.0 if p is None else float(p)
            p = max(p, 0.0)
            w = float(np.log1p(p))
            w = min(w, cap)

            # if repeated (u,a), keep max
            if w > train_w_by_user[u].get(a, 0.0):
                train_w_by_user[u][a] = w

        # Test positives unchanged
        for u, a in zip(test_tbl["author_steamid"].to_pylist(), test_tbl["appid"].to_pylist()):
            if len(test_items_by_user[u]) < max_test_per_user:
                test_items_by_user[u].add(a)

        if rg % 25 == 0:
            print(f"[playtime] scanned row group {rg}/{pf.num_row_groups}")

    users_final_w = [u for u, d in train_w_by_user.items() if len(d) >= MIN_TRAIN_POS]
    n_eval = sum(1 for u in users_final_w if len(test_items_by_user.get(u, set())) > 0)

    print("users_final (>= MIN_TRAIN_POS weighted train):", len(users_final_w))
    print("users_final with any test positives:", n_eval)

    return train_w_by_user, test_items_by_user, users_final_w

def build_csr_from_weights(train_w_by_user: dict, users_final: list):
    user2idx = {u: i for i, u in enumerate(users_final)}
    item_set = set()
    for u in users_final:
        item_set.update(train_w_by_user[u].keys())
    items = list(item_set)
    item2idx = {a: i for i, a in enumerate(items)}
    idx2item = {i: a for a, i in item2idx.items()}

    rows, cols, vals = [], [], []
    for u in users_final:
        ui = user2idx[u]
        for a, w in train_w_by_user[u].items():
            ii = item2idx.get(a)
            if ii is not None:
                rows.append(ui); cols.append(ii); vals.append(float(w))

    X = csr_matrix((vals, (rows, cols)), shape=(len(users_final), len(items))).tocsr()
    print("X_weighted shape:", X.shape, "| nnz:", X.nnz, "| w min/max:", (min(vals), max(vals)) if vals else (None, None))
    return X, user2idx, item2idx, idx2item

train_w_play, test_play, users_play = build_train_test_playtime_weights(
    DATA_PATH, sampled_users, TRAIN_END_YEAR, MAX_TRAIN_PER_USER, MAX_TEST_PER_USER,
    PLAYTIME_COL, cap=PLAYTIME_CAP
)

X_play, u2i_play, i2i_play, idx2_play = build_csr_from_weights(train_w_play, users_play)

cfg = ALS_DEFAULT.copy()
m_play = AlternatingLeastSquares(
    factors=cfg["factors"],
    regularization=cfg["regularization"],
    iterations=cfg["iterations"],
)
m_play.fit(X_play * cfg["alpha"])

hr, rec, prec, n_eval = eval_topk_implicit(
    m_play, X_play, users_play, u2i_play, idx2_play, test_play, K=K
)
print(f"Playtime-weighted ALS | HR@{K}={hr:.6f} Recall@{K}={rec:.6f} Prec@{K}={prec:.6f} | Users={n_eval}")


[playtime] scanned row group 0/165
[playtime] scanned row group 25/165
[playtime] scanned row group 50/165
[playtime] scanned row group 75/165
[playtime] scanned row group 100/165
[playtime] scanned row group 125/165
[playtime] scanned row group 150/165
users_final (>= MIN_TRAIN_POS weighted train): 10779
users_final with any test positives: 6189
X_weighted shape: (10779, 10022) | nnz: 109903 | w min/max: (0.6931471805599453, 8.0)


  0%|          | 0/20 [00:00<?, ?it/s]

Playtime-weighted ALS | HR@10=0.148005 Recall@10=0.043146 Prec@10=0.018452 | Users=6189


## 11) Feature-weighted ALS: playtime + sentiment (Testing)

We also test incorporating review-text sentiment as a weak confidence modifier on top of playtime:

- compute VADER compound sentiment on the review text
- keep only positive sentiment contribution
- combine: `w = log1p(playtime) * (1 + max(sentiment, 0))`

This is more expensive to run (text processing), so we treat it as an ablation.


In [38]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

SENT_COL = "review"
PLAYTIME_CAP = 8.0

def build_train_test_playtime_sent_weights(
    data_path: str,
    sampled_users: set,
    train_end_year: int,
    max_train_per_user: int,
    max_test_per_user: int,
    playtime_col: str,
    sent_col: str,
    cap: float = 8.0,
):
    pf = pq.ParquetFile(data_path)
    train_w_by_user = defaultdict(dict)
    test_items_by_user = defaultdict(set)
    sent_cache = {}

    for rg in range(pf.num_row_groups):
        tbl = pf.read_row_group(
            rg,
            columns=["author_steamid", "appid", "voted_up", "timestamp_created", playtime_col, sent_col],
        )
        users = tbl["author_steamid"].to_pylist()
        keep_mask = pa.array([u in sampled_users for u in users])
        tbl = tbl.filter(keep_mask)
        if tbl.num_rows == 0:
            continue

        years = pc.year(tbl["timestamp_created"])
        is_pos = pc.equal(tbl["voted_up"], True)

        train_tbl = tbl.filter(pc.and_(pc.less(years, train_end_year), is_pos))
        test_tbl  = tbl.filter(pc.and_(pc.greater_equal(years, train_end_year), is_pos))

        u_list = train_tbl["author_steamid"].to_pylist()
        a_list = train_tbl["appid"].to_pylist()
        p_list = train_tbl[playtime_col].to_pylist()
        t_list = train_tbl[sent_col].to_pylist()

        for u, a, p, text in zip(u_list, a_list, p_list, t_list):
            if (a not in train_w_by_user[u]) and (len(train_w_by_user[u]) >= max_train_per_user):
                continue

            p = 0.0 if p is None else float(p)
            p = max(p, 0.0)
            play_w = float(np.log1p(p))

            if text is None or text == "":
                sent = 0.0
            else:
                s = sent_cache.get(text)
                if s is None:
                    s = analyzer.polarity_scores(text)["compound"]
                    sent_cache[text] = s
                sent = s

            sent_w = max(float(sent), 0.0)
            w = play_w * (1.0 + sent_w)
            w = min(w, cap)

            if w > train_w_by_user[u].get(a, 0.0):
                train_w_by_user[u][a] = w

        for u, a in zip(test_tbl["author_steamid"].to_pylist(), test_tbl["appid"].to_pylist()):
            if len(test_items_by_user[u]) < max_test_per_user:
                test_items_by_user[u].add(a)

        if rg % 25 == 0:
            print(f"[sent] scanned row group {rg}/{pf.num_row_groups}")

    users_final_w = [u for u, d in train_w_by_user.items() if len(d) >= MIN_TRAIN_POS]
    n_eval = sum(1 for u in users_final_w if len(test_items_by_user.get(u, set())) > 0)
    print("users_final:", len(users_final_w), "| eval users:", n_eval)

    return train_w_by_user, test_items_by_user, users_final_w

train_w_sent, test_sent, users_sent = build_train_test_playtime_sent_weights(
    DATA_PATH, sampled_users, TRAIN_END_YEAR, MAX_TRAIN_PER_USER, MAX_TEST_PER_USER,
    PLAYTIME_COL, SENT_COL, cap=PLAYTIME_CAP
)
X_sent, u2i_sent, i2i_sent, idx2_sent = build_csr_from_weights(train_w_sent, users_sent)

cfg = ALS_DEFAULT.copy()
m_sent = AlternatingLeastSquares(
    factors=cfg["factors"],
    regularization=cfg["regularization"],
    iterations=cfg["iterations"],
)
m_sent.fit(X_sent * cfg["alpha"])

hr, rec, prec, n_eval = eval_topk_implicit(
    m_sent, X_sent, users_sent, u2i_sent, idx2_sent, test_sent, K=K
)
print(f"Playtime+Sentiment ALS | HR@{K}={hr:.6f} Recall@{K}={rec:.6f} Prec@{K}={prec:.6f} | Users={n_eval}")


[sent] scanned row group 0/165
[sent] scanned row group 25/165
[sent] scanned row group 50/165
[sent] scanned row group 75/165
[sent] scanned row group 100/165
[sent] scanned row group 125/165
[sent] scanned row group 150/165
users_final: 10779 | eval users: 6189
X_weighted shape: (10779, 10022) | nnz: 109903 | w min/max: (0.6931471805599453, 8.0)


  0%|          | 0/20 [00:00<?, ?it/s]

Playtime+Sentiment ALS | HR@10=0.142996 Recall@10=0.041182 Prec@10=0.017612 | Users=6189


## 12) Testing: Negative weighting (pos + neg in train)

To use more than the `voted_up=True` signal:
- Train includes both positives and negatives (before cutoff)
- Positives get weight 1.0
- Negatives get a smaller nonnegative weight (e.g., 0.1–0.3)
- Test remains **future positives only**

This keeps implicit ALS valid (nonnegative weights) while incorporating weak “dislike” evidence.


In [39]:
POS_W = 1.0
NEG_W = 0.2  # try 0.1, 0.2, 0.3

def build_train_test_posneg_weights(
    data_path: str,
    sampled_users: set,
    train_end_year: int,
    max_train_per_user: int,
    max_test_per_user: int,
    pos_w: float,
    neg_w: float,
):
    pf = pq.ParquetFile(data_path)
    train_w_by_user = defaultdict(dict)   # u -> {appid: weight}
    test_items_by_user = defaultdict(set) # positives only

    for rg in range(pf.num_row_groups):
        tbl = pf.read_row_group(
            rg,
            columns=["author_steamid", "appid", "voted_up", "timestamp_created"]
        )

        users = tbl["author_steamid"].to_pylist()
        keep_mask = pa.array([u in sampled_users for u in users])
        tbl = tbl.filter(keep_mask)
        if tbl.num_rows == 0:
            continue

        years = pc.year(tbl["timestamp_created"])

        # Train = all reviews before cutoff (pos+neg)
        train_tbl = tbl.filter(pc.less(years, train_end_year))

        u_list = train_tbl["author_steamid"].to_pylist()
        a_list = train_tbl["appid"].to_pylist()
        v_list = train_tbl["voted_up"].to_pylist()

        for u, a, v in zip(u_list, a_list, v_list):
            if (a not in train_w_by_user[u]) and (len(train_w_by_user[u]) >= max_train_per_user):
                continue
            w = pos_w if v else neg_w
            if w > train_w_by_user[u].get(a, 0.0):
                train_w_by_user[u][a] = w

        # Test = positives after cutoff
        is_pos = pc.equal(tbl["voted_up"], True)
        test_tbl = tbl.filter(pc.and_(pc.greater_equal(years, train_end_year), is_pos))
        for u, a in zip(test_tbl["author_steamid"].to_pylist(), test_tbl["appid"].to_pylist()):
            if len(test_items_by_user[u]) < max_test_per_user:
                test_items_by_user[u].add(a)

        if rg % 25 == 0:
            print(f"[posneg] scanned row group {rg}/{pf.num_row_groups}")

    users_final = [u for u, d in train_w_by_user.items() if len(d) >= MIN_TRAIN_POS]
    n_eval = sum(1 for u in users_final if len(test_items_by_user.get(u, set())) > 0)
    print("users_final:", len(users_final), "| eval users:", n_eval)
    return train_w_by_user, test_items_by_user, users_final

train_w_by_user, test_pos_only, users_final_neg = build_train_test_posneg_weights(
    DATA_PATH, sampled_users, TRAIN_END_YEAR,
    MAX_TRAIN_PER_USER, MAX_TEST_PER_USER,
    POS_W, NEG_W
)

# Build CSR from weights
user2idx_neg = {u:i for i,u in enumerate(users_final_neg)}

item_set = set()
for u in users_final_neg:
    item_set.update(train_w_by_user[u].keys())
items_neg = list(item_set)

item2idx_neg = {a:i for i,a in enumerate(items_neg)}
idx2item_neg = {i:a for a,i in item2idx_neg.items()}

rows, cols, vals = [], [], []
for u in users_final_neg:
    ui = user2idx_neg[u]
    for a, w in train_w_by_user[u].items():
        ii = item2idx_neg.get(a)
        if ii is not None:
            rows.append(ui); cols.append(ii); vals.append(float(w))

X_neg = csr_matrix((vals, (rows, cols)), shape=(len(users_final_neg), len(items_neg))).tocsr()
print("X_neg:", X_neg.shape, "nnz:", X_neg.nnz, "min/max w:", (min(vals), max(vals)))


[posneg] scanned row group 0/165
[posneg] scanned row group 25/165
[posneg] scanned row group 50/165
[posneg] scanned row group 75/165
[posneg] scanned row group 100/165
[posneg] scanned row group 125/165
[posneg] scanned row group 150/165
users_final: 12919 | eval users: 7064
X_neg: (12919, 12926) nnz: 140532 min/max w: (0.2, 1.0)


In [40]:
from implicit.nearest_neighbours import bm25_weight

X_neg_bm25 = bm25_weight(X_neg, K1=BM25_DEFAULT["K1"], B=BM25_DEFAULT["B"]).tocsr()

cfg = ALS_DEFAULT.copy()
m_neg_bm25 = AlternatingLeastSquares(
    factors=cfg["factors"],
    regularization=cfg["regularization"],
    iterations=cfg["iterations"],
)
m_neg_bm25.fit(X_neg_bm25 * cfg["alpha"])

hr, rec, prec, n_eval = eval_topk_implicit(
    m_neg_bm25, X_neg_bm25, users_final_neg, user2idx_neg, idx2item_neg, test_pos_only, K=K
)
print(f"NegWeight+BM25+ALS | HR@{K}={hr:.6f} Recall@{K}={rec:.6f} Prec@{K}={prec:.6f} | Users={n_eval}")


  0%|          | 0/20 [00:00<?, ?it/s]

NegWeight+BM25+ALS | HR@10=0.157135 Recall@10=0.048555 Prec@10=0.019734 | Users=7064


## 13) Sweeps + multi-seed stability

To avoid one lucky run, we evaluate across multiple seeds and report stability.


In [41]:
def run_seed_experiment(seed: int, use_bm25: bool = True, cfg=None):
    if cfg is None:
        cfg = ALS_DEFAULT.copy()

    sampled = sample_users(eligible_users, TARGET_USERS, seed)

    train_items, test_items, users_f = build_train_test_pos_only(
        DATA_PATH, sampled, TRAIN_END_YEAR, MAX_TRAIN_PER_USER, MAX_TEST_PER_USER
    )
    X, u2i, i2i, idx2 = build_csr_binary(train_items, users_f)

    X_train = X
    if use_bm25:
        X_train = bm25_weight(X, K1=BM25_DEFAULT["K1"], B=BM25_DEFAULT["B"]).tocsr()

    m = AlternatingLeastSquares(
        factors=cfg["factors"],
        regularization=cfg["regularization"],
        iterations=cfg["iterations"],
    )
    m.fit(X_train * cfg["alpha"])
    hr, rec, prec, n_eval = eval_topk_implicit(m, X_train, users_f, u2i, idx2, test_items, K=K)
    return {"seed": seed, "hr": hr, "recall": rec, "precision": prec, "users": n_eval}

results = [run_seed_experiment(s, use_bm25=True) for s in SEEDS]
results


[split] scanned row group 0/165
[split] scanned row group 25/165
[split] scanned row group 50/165
[split] scanned row group 75/165
[split] scanned row group 100/165
[split] scanned row group 125/165
[split] scanned row group 150/165
users_final (>= MIN_TRAIN_POS train positives): 10928
users_final with any test positives: 6254
X_user_item shape: (10928, 11155) | nnz: 112692


  0%|          | 0/20 [00:00<?, ?it/s]

[split] scanned row group 0/165
[split] scanned row group 25/165
[split] scanned row group 50/165
[split] scanned row group 75/165
[split] scanned row group 100/165
[split] scanned row group 125/165
[split] scanned row group 150/165
users_final (>= MIN_TRAIN_POS train positives): 10963
users_final with any test positives: 6267
X_user_item shape: (10963, 11460) | nnz: 114066


  0%|          | 0/20 [00:00<?, ?it/s]

[split] scanned row group 0/165
[split] scanned row group 25/165
[split] scanned row group 50/165
[split] scanned row group 75/165
[split] scanned row group 100/165
[split] scanned row group 125/165
[split] scanned row group 150/165
users_final (>= MIN_TRAIN_POS train positives): 11109
users_final with any test positives: 6324
X_user_item shape: (11109, 11758) | nnz: 116854


  0%|          | 0/20 [00:00<?, ?it/s]

[{'seed': 232,
  'hr': 0.15206267988487368,
  'recall': 0.04400886185272802,
  'precision': 0.01874000639590662,
  'users': 6254},
 {'seed': 777,
  'hr': 0.17249082495611937,
  'recall': 0.04830795872718089,
  'precision': 0.021365884793362055,
  'users': 6267},
 {'seed': 2026,
  'hr': 0.16492726122707146,
  'recall': 0.04860385516123221,
  'precision': 0.021331435800126503,
  'users': 6324}]

In [42]:
hrs = [r["hr"] for r in results]
print("BM25+ALS HR@10 across seeds:", hrs)
print("Mean:", float(np.mean(hrs)), "Std:", float(np.std(hrs)))


BM25+ALS HR@10 across seeds: [0.15206267988487368, 0.17249082495611937, 0.16492726122707146]
Mean: 0.1631602553560215 Std: 0.008432833057923401


In [43]:
from implicit.als import AlternatingLeastSquares
import numpy as np

def run_seed_experiment_raw_als(seed: int, cfg=None):
    if cfg is None:
        cfg = ALS_DEFAULT.copy()

    sampled = sample_users(eligible_users, TARGET_USERS, seed)

    train_items, test_items, users_f = build_train_test_pos_only(
        DATA_PATH, sampled, TRAIN_END_YEAR, MAX_TRAIN_PER_USER, MAX_TEST_PER_USER
    )

    X, u2i, i2i, idx2 = build_csr_binary(train_items, users_f)

    m = AlternatingLeastSquares(
        factors=cfg["factors"],
        regularization=cfg["regularization"],
        iterations=cfg["iterations"],
    )

    # Raw ALS: no BM25/TFIDF weighting, just confidence scaling
    m.fit(X * cfg["alpha"])

    hr, rec, prec, n_eval = eval_topk_implicit(
        m, X, users_f, u2i, idx2, test_items, K=K
    )

    return {
        "seed": seed,
        "hr": float(hr),
        "recall": float(rec),
        "precision": float(prec),
        "users": int(n_eval),
        "n_users_final": int(len(users_f)),
        "n_items": int(X.shape[1]),
        "nnz": int(X.nnz),
    }

raw_results = [run_seed_experiment_raw_als(s) for s in SEEDS]
raw_results


[split] scanned row group 0/165
[split] scanned row group 25/165
[split] scanned row group 50/165
[split] scanned row group 75/165
[split] scanned row group 100/165
[split] scanned row group 125/165
[split] scanned row group 150/165
users_final (>= MIN_TRAIN_POS train positives): 10928
users_final with any test positives: 6254
X_user_item shape: (10928, 11155) | nnz: 112692


  0%|          | 0/20 [00:00<?, ?it/s]

[split] scanned row group 0/165
[split] scanned row group 25/165
[split] scanned row group 50/165
[split] scanned row group 75/165
[split] scanned row group 100/165
[split] scanned row group 125/165
[split] scanned row group 150/165
users_final (>= MIN_TRAIN_POS train positives): 10963
users_final with any test positives: 6267
X_user_item shape: (10963, 11460) | nnz: 114066


  0%|          | 0/20 [00:00<?, ?it/s]

[split] scanned row group 0/165
[split] scanned row group 25/165
[split] scanned row group 50/165
[split] scanned row group 75/165
[split] scanned row group 100/165
[split] scanned row group 125/165
[split] scanned row group 150/165
users_final (>= MIN_TRAIN_POS train positives): 11109
users_final with any test positives: 6324
X_user_item shape: (11109, 11758) | nnz: 116854


  0%|          | 0/20 [00:00<?, ?it/s]

[{'seed': 232,
  'hr': 0.1626159258074832,
  'recall': 0.04776388419511096,
  'precision': 0.020338983050847456,
  'users': 6254,
  'n_users_final': 10928,
  'n_items': 11155,
  'nnz': 112692},
 {'seed': 777,
  'hr': 0.1814265198659646,
  'recall': 0.05187964499603146,
  'precision': 0.022945588000638263,
  'users': 6267,
  'n_users_final': 10963,
  'n_items': 11460,
  'nnz': 114066},
 {'seed': 2026,
  'hr': 0.17773561037318153,
  'recall': 0.05271122320302648,
  'precision': 0.023134092346616064,
  'users': 6324,
  'n_users_final': 11109,
  'n_items': 11758,
  'nnz': 116854}]

In [48]:
summary_rows = []

def add_row(name, hr, rec, prec, users):
    summary_rows.append({
        "model": name,
        f"HR@{K}": float(hr),
        f"Recall@{K}": float(rec),
        f"Prec@{K}": float(prec),
        "eval_users": int(users),
    })

# From existing outputs in notebook:
add_row("Popularity", hrp, recp, precp, n_pop)
add_row("ALS (default)", hr, rec, prec, n_eval)
add_row("BM25 + ALS", hr, rec, prec, n_eval)
add_row("ALS (best tuned)", best_hr, best_rec, best_prec, best_users)
#add_row("Playtime-weighted ALS", hr_play, rec_play, prec_play, n_eval_play)
#add_row("Playtime+Sentiment ALS", hr_sent, rec_sent, prec_sent, n_eval_sent)

df_summary = pd.DataFrame(summary_rows).sort_values(by=f"HR@{K}", ascending=False)
df_summary


,model,HR@10,Recall@10,Prec@10,eval_users
3,ALS (best tuned),0.162936,0.047614,0.020275,6254
1,ALS (default),0.157135,0.048555,0.019734,7064
2,BM25 + ALS,0.157135,0.048555,0.019734,7064
0,Popularity,0.111289,0.033457,0.014247,6254


In [49]:
import numpy as np

def catalog_coverage_at_k(model, X, users_final, user2idx, idx2item, K=10):
    rec_items = set()
    users_evaluated = 0

    for u in users_final:
        ui = user2idx.get(u)
        if ui is None:
            continue

        ids, scores = model.recommend(
            np.array([int(ui)]),
            X[[int(ui)]],
            N=K,
            filter_already_liked_items=True,
        )
        for i in ids[0]:
            rec_items.add(idx2item[int(i)])
        users_evaluated += 1

    catalog_size = len(idx2item)
    coverage = len(rec_items) / catalog_size if catalog_size else 0.0
    return coverage, len(rec_items), catalog_size, users_evaluated

cov, n_unique, n_catalog, n_users = catalog_coverage_at_k(
    m_als, X_user_item, users_final, user2idx, idx2item, K=K
)
print(f"Catalog coverage@{K}: {cov:.4f} ({n_unique} unique recommended items / {n_catalog} catalog items) | users={n_users}")


Catalog coverage@10: 0.0522 (582 unique recommended items / 11155 catalog items) | users=10928


In [57]:
def compute_coverage_for_models(models_dict, K=10):
    rows = []
    for name, (model, X) in models_dict.items():
        cov, n_unique, n_catalog, n_users = catalog_coverage_at_k(
            model, X, users_final, user2idx, idx2item, K=K
        )
        rows.append({
            "model": name,
            f"coverage@{K}": cov,
            "unique_items_recommended": n_unique,
            "catalog_size": n_catalog,
            "users": n_users
        })
    return pd.DataFrame(rows).sort_values(by=f"coverage@{K}", ascending=False)

models_dict = {
    "ALS (default)": (m_als, X_user_item),
    "ALS (best tuned)": (best_model_raw, X_user_item),
    "BM25 + ALS": (m_bm25, X_bm25),
    "BM25 + ALS (best)": (m_bm25_best, X_bm25_best),
}

df_cov = compute_coverage_for_models(models_dict, K=K)
df_cov


,model,coverage@10,unique_items_recommended,catalog_size,users
2,BM25 + ALS,0.104169,1162,11155,10928
3,BM25 + ALS (best),0.062931,702,11155,10928
0,ALS (default),0.052174,582,11155,10928
1,ALS (best tuned),0.037113,414,11155,10928


## 14) Save pretrained artifacts (optional)

We can save:
- user_factors.npy
- item_factors.npy
- mappings (user2idx, idx2item)
- config

This allows fast demo/reload without retraining.


In [ ]:
ART_DIR = "artifacts_bm25_als"
os.makedirs(ART_DIR, exist_ok=True)

# Choose which model to save (here: m_bm25 from earlier section)
np.save(os.path.join(ART_DIR, "user_factors.npy"), m_bm25.user_factors)
np.save(os.path.join(ART_DIR, "item_factors.npy"), m_bm25.item_factors)

with open(os.path.join(ART_DIR, "user2idx.pkl"), "wb") as f:
    pickle.dump(user2idx, f)

with open(os.path.join(ART_DIR, "idx2item.pkl"), "wb") as f:
    pickle.dump(idx2item, f)

with open(os.path.join(ART_DIR, "config.json"), "w") as f:
    json.dump(
        {
            "DATA_PATH": DATA_PATH,
            "MIN_REVIEWS": MIN_REVIEWS,
            "TARGET_USERS": TARGET_USERS,
            "TRAIN_END_YEAR": TRAIN_END_YEAR,
            "MIN_TRAIN_POS": MIN_TRAIN_POS,
            "K": K,
            "MAX_TRAIN_PER_USER": MAX_TRAIN_PER_USER,
            "MAX_TEST_PER_USER": MAX_TEST_PER_USER,
            "ALS_DEFAULT": ALS_DEFAULT,
            "BM25_DEFAULT": BM25_DEFAULT,
            "seed": SEEDS[0],
        },
        f,
        indent=2,
    )

print("Saved artifacts to:", ART_DIR)
